# NT-Cas12a: Colab execution handoff (#109)

**Implementation ready; actual production Colab training is unexecuted.**
This notebook calls `cas12a.nt_workflow.NTWorkflow`; it contains no data,
training, evaluation or checkpoint-recovery implementation.
Read [the runbook](https://github.com/shin584/project/blob/model_analysis/cas12a/docs/nt-colab.md)
before execution. Offline fixture results establish code readiness only.

**Human boundary 1:** open a Colab session, choose a GPU runtime and connect.
**Human boundary 2:** authorize Drive when prompted below.
**Human boundary 3:** configure source/locations and explicitly start fresh or
resume training. Execute cells individually, inspecting each stage. No actual
run, extra fit, GPU purchase or Windows GPU provisioning is automated here.

## 1. Retrieve the reviewed source and install the fixed dependency recipe

Set `CODE_REF` to the commit containing this notebook (publish that commit or
upload its checkout first). Keep the checkout on the VM under `/content`.
Use Python 3.11–3.13. The pinned versions are verified with CPU fixtures;
the CUDA 12.6 wheel is a Colab candidate until production preflight passes.
Installation changes this disposable session's packages. Optional vision/audio
torch extensions are removed to avoid mismatches with its fixed torch version.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/shin584/project.git"
CODE_REF = ""  # Required: reviewed commit SHA containing #109.
REPO_ROOT = Path("/content/MC_project")
if not CODE_REF:
    raise ValueError("Set CODE_REF to the reviewed implementation commit")
if not REPO_ROOT.exists():
    subprocess.check_call(["git", "clone", "--no-checkout", REPO_URL, str(REPO_ROOT)])
subprocess.check_call(["git", "fetch", "origin", CODE_REF], cwd=REPO_ROOT)
subprocess.check_call(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO_ROOT)
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
print(subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())

In [ ]:
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "uninstall",
        "-y",
        "torchvision",
        "torchaudio",
        "torchtext",
    ]
)
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "torch==2.12.0+cu126",
        "--index-url",
        "https://download.pytorch.org/whl/cu126",
    ]
)
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-r",
        "cas12a/requirements-nt.txt",
    ]
)

**After installation:** restart the Python session if Colab requests it or
any scientific package was already imported. Re-run `source-setup`, skip
`install`, and continue below. Do not reinstall or restart during a run:
resume requires the same VM, GPU and frozen runtime environment.

In [ ]:
from google.colab import drive

drive_mount_root = Path("/content/drive")
drive.mount(str(drive_mount_root))  # Human Drive authorization prompt.

## 2. Configure the workflow

Supply the original Kim 2018 source (15,000 usable targets). No source data
or production weights are bundled. The source may be on Drive; checkpoints
and transient work must remain under `/content`, outside the Drive mount.
Keep the reusable partition outside run directories, preferably on Drive.
Production split/seed/recipe are fixed; do not edit them after final results.

Choose `mode="fresh"` and a new `run_id`, or `mode="resume"` and the original
`resume_run_id` with the identical source, partition and roots. Resume skips
prepare/preflight and only continues an eligible same-VM training checkpoint.
`mode="choose"` prevents an accidental unconfigured Run all.

In [ ]:
from cas12a.nt_workflow import ModelSource, NTWorkflow, SplitConfig

source = Path("/content/drive/MyDrive/cas12a-nt/source.xlsx")
partition_path = Path("/content/drive/MyDrive/cas12a-nt/partition.json")
local_work_root = Path("/content/cas12a-nt-work")
drive_output_root = Path("/content/drive/MyDrive/cas12a-nt/runs")
split = SplitConfig()  # Production: 12000/1500/1500, seed 42.
model_source = ModelSource(revision="main")  # Preflight freezes the resolved SHA.
mode = "choose"  # Set explicitly to "fresh" or "resume".
run_id = ""  # Required fresh ID; existing destinations are refused.
resume_run_id = ""  # Required original ID in resume mode.

if mode not in ("fresh", "resume"):
    raise ValueError("Explicitly choose fresh or resume")
if not (run_id if mode == "fresh" else resume_run_id):
    raise ValueError("Supply a fresh run_id or the original resume_run_id")
if not local_work_root.resolve().is_relative_to(Path("/content")):
    raise ValueError("Colab work/checkpoints must use the VM's /content")
if not drive_output_root.resolve().is_relative_to(drive_mount_root.resolve()):
    raise ValueError("Durable output must be beneath the authorized Drive mount")

In [ ]:
workflow = NTWorkflow(
    source=source,
    partition_path=partition_path,
    local_work_root=local_work_root,
    drive_output_root=drive_output_root,
    drive_mount_root=drive_mount_root,
    split=split,
)

In [ ]:
import importlib.metadata
import json
import platform
from datetime import datetime, timezone
from uuid import uuid4

import torch

session_evidence = {
    "recorded_utc": datetime.now(timezone.utc).isoformat(),
    "code_commit": subprocess.check_output(
        ["git", "rev-parse", "HEAD"], text=True
    ).strip(),
    "python": platform.python_version(),
    "pip_freeze": subprocess.check_output(
        [sys.executable, "-m", "pip", "freeze"], text=True
    ).splitlines(),
    "packages": {
        name: importlib.metadata.version(name)
        for name in (
            "torch",
            "transformers",
            "tokenizers",
            "huggingface-hub",
            "safetensors",
            "numpy",
            "pandas",
            "scipy",
            "openpyxl",
        )
    },
    "cuda_runtime": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "mode": mode,
    "run_id": run_id if mode == "fresh" else resume_run_id,
    "production_training_completed": False,
}
session_name = f"nt-session-{uuid4().hex}.json"
local_session = Path("/content") / session_name
local_session.write_text(json.dumps(session_evidence, indent=2), encoding="utf-8")
session_directory = drive_output_root.parent / "sessions"
session_directory.mkdir(parents=True, exist_ok=True)
durable_session = session_directory / session_name
durable_session.write_bytes(local_session.read_bytes())
assert durable_session.read_bytes() == local_session.read_bytes()
print(session_evidence, durable_session)

## 3. Prepare → production preflight (fresh only)

Preparation validates data and verifies or creates the immutable partition.
Preflight downloads the original NT-500M model/tokenizer, checks full-input
tokenization, runs two optimizer updates (including backward after Adam-state
allocation) and validation, then freezes feasible batch/precision settings.
A passed preflight is **not completed training**. Resource failure stops;
inspect evidence rather than changing the model or using Testset values.

In [ ]:
if mode == "fresh":
    run = workflow.prepare(run_id=run_id)
    print(run.status, run.local_dir, run.durable_dir, run.checkpoint_dir)

In [ ]:
if mode == "fresh":
    run = workflow.preflight(run, model=model_source)
    print(run.status)

## 4. Human execution start: fit/select OR explicit same-VM resume

Execute this cell only when ready to start the bounded training run.
`fit` uses fresh state, validation-MSE selection and the fixed recipe;
`resume` verifies and restores complete surviving local training state.
Neither operation silently starts another run on failure. On lost local
state, preserve the old evidence and explicitly configure a new fresh ID
using the same verified partition. Never resume from a selected export.

In [ ]:
if mode == "fresh":
    run = workflow.fit(run)
elif mode == "resume":
    run = workflow.resume(resume_run_id)
else:
    raise ValueError("Explicitly choose fresh or resume")
print(run.status)

## 5. Local fresh-instance reload → verified durable selected export

Training completion is separate from local reload and durable validation.
Only selected weights/config/tokenizer and evidence go to Drive. Routine
optimizer/RNG checkpoints stay in `/content`; they may vanish with the VM.
Existing export/evaluation destinations are preserved; do not replay cells
after a completed stage or delete evidence to force a retry.

In [ ]:
run = workflow.export_selected(run)
run = workflow.verify_export(run)
print(run.status)

In [ ]:
run = workflow.persist_export(run)
run = workflow.verify_durable_export(run)
print(run.status)

## 6. Frozen final Testset evaluation → verified durable report

Evaluate once on the 1,500 reserved targets. Keep raw/clipped metrics and
the training-only mean baseline. Valid low metrics are reportable; no CNN
superiority gate applies. Do not use final results to select settings or
trigger another fit. Nonfinite/raw-collapse or identity failures stop
completion. Historical CNN five-fold results are not directly comparable.

In [ ]:
run = workflow.evaluate(run)
print(run.status)

In [ ]:
run = workflow.persist_evaluation(run)
run = workflow.verify_durable_evaluation(run)
print(run.status, run.durable_dir / "final_evaluation")

## 7. Handoff and evidence inspection

Use the [runbook artifact checklist](https://github.com/shin584/project/blob/model_analysis/cas12a/docs/nt-colab.md)
to inspect `status.json`, `lifecycle.json`, preflight/execution, selected
history/reload, both durable receipts and the final metrics/predictions.
Preserve this notebook with actual outputs, session JSON, run ID, code SHA
and durable destination. Verify Drive-copy hashes independently as described
in the runbook. Only a production run with all required evidence verified
can claim `model_completed: true`. Fixture completion never claims that.

On interruption, OOM, disk or Drive failure: stop, preserve traceback and
both status copies, inspect the runbook recovery table, and explicitly
choose the permitted next operation. No automatic retries or extra fits.